# ⚾ HR Scout — MLB Home Run Prediction Pipeline

**Daily workflow:**
1. Run **Cell 1** once to install dependencies
2. Run **Cell 2** to set the game date
3. Run **Cell 3** to fetch all live data (lineups, pitchers, Statcast, weather)
4. Run **Cell 4** to score all batters with the built-in statistical model
5. Run **Cell 5** to save `predictions.json` and push to GitHub Pages
6. After games: Run **Cell 6** to log results
7. Run **Cell 7** any time for the calibration report

No AI API required — scoring is done entirely in Python from live MLB/Statcast data.

In [ ]:
# ── CELL 1: Install dependencies (run once) ───────────────────────────────────
import sys
!{sys.executable} -m pip install requests pandas pybaseball --quiet
print('✓ Dependencies ready')

In [ ]:
# ── CELL 2: Configuration ─────────────────────────────────────────────────────
from datetime import date, timedelta

GAME_DATE      = 'today'   # or e.g. '2026-04-10'
USE_STATCAST   = True      # set False for faster run
TOP_N_PLAYERS  = 45        # how many players to rank
REPO_PATH      = '.'       # path to your local GitHub Pages repo
AUTO_PUSH      = True      # set False to skip git push

SC_START_2025  = '2025-03-01'
SC_END_2025    = '2025-10-01'
SC_START_2026  = '2026-03-27'
SC_END_2026    = (date.today() - timedelta(days=1)).isoformat()
SC_MIN_PA_2026 = 50

if GAME_DATE == 'today':
    GAME_DATE = date.today().isoformat()

print(f'✓ Game date      : {GAME_DATE}')
print(f'  Statcast 2026  : {SC_START_2026} → {SC_END_2026}')
print(f'  Statcast 2025  : {SC_START_2025} → {SC_END_2025}')
print(f'  Auto-push      : {AUTO_PUSH}')

In [ ]:
# ── CELL 3: Fetch all live data ───────────────────────────────────────────────
import requests, json, time, math
import pandas as pd
from datetime import datetime

try:
    from pybaseball import statcast_batter, statcast_pitcher, playerid_lookup
    PYBASEBALL = True
except ImportError:
    PYBASEBALL = False
    print('⚠ pybaseball not available — Statcast features disabled')

PARKS = {
    133: {'name':'Oakland Coliseum',         'hr_L':0.93,'hr_R':0.91,'lat':37.752,'lon':-122.201,'roof':False,'orientation':200},
    134: {'name':'PNC Park',                 'hr_L':0.96,'hr_R':0.94,'lat':40.447,'lon':-80.006, 'roof':False,'orientation':100},
    135: {'name':'Petco Park',               'hr_L':0.89,'hr_R':0.85,'lat':32.707,'lon':-117.157,'roof':False,'orientation':300},
    136: {'name':'T-Mobile Park',            'hr_L':0.93,'hr_R':0.89,'lat':47.591,'lon':-122.333,'roof':True, 'orientation':0},
    137: {'name':'Oracle Park',              'hr_L':0.85,'hr_R':0.80,'lat':37.778,'lon':-122.389,'roof':False,'orientation':55},
    138: {'name':'Busch Stadium',            'hr_L':0.98,'hr_R':0.96,'lat':38.623,'lon':-90.193, 'roof':False,'orientation':68},
    139: {'name':'Tropicana Field',          'hr_L':1.00,'hr_R':0.97,'lat':27.768,'lon':-82.653, 'roof':True, 'orientation':0},
    140: {'name':'Globe Life Field',         'hr_L':0.96,'hr_R':0.99,'lat':32.748,'lon':-97.084, 'roof':True, 'orientation':0},
    141: {'name':'Rogers Centre',            'hr_L':1.01,'hr_R':1.03,'lat':43.641,'lon':-79.389, 'roof':True, 'orientation':0},
    142: {'name':'Target Field',             'hr_L':0.94,'hr_R':0.96,'lat':44.982,'lon':-93.278, 'roof':False,'orientation':320},
    143: {'name':'Citizens Bank Park',       'hr_L':1.14,'hr_R':1.10,'lat':39.906,'lon':-75.166, 'roof':False,'orientation':62},
    144: {'name':'Truist Park',              'hr_L':1.05,'hr_R':1.09,'lat':33.891,'lon':-84.468, 'roof':False,'orientation':15},
    145: {'name':'Guaranteed Rate Field',    'hr_L':1.02,'hr_R':1.04,'lat':41.830,'lon':-87.634, 'roof':False,'orientation':5},
    146: {'name':'LoanDepot Park',           'hr_L':0.90,'hr_R':0.86,'lat':25.778,'lon':-80.220, 'roof':True, 'orientation':0},
    147: {'name':'Yankee Stadium',           'hr_L':1.10,'hr_R':1.01,'lat':40.829,'lon':-73.926, 'roof':False,'orientation':42},
    158: {'name':'American Family Field',    'hr_L':1.08,'hr_R':1.04,'lat':43.028,'lon':-87.971, 'roof':True, 'orientation':0},
    108: {'name':'Angel Stadium',            'hr_L':0.94,'hr_R':0.98,'lat':33.800,'lon':-117.883,'roof':False,'orientation':340},
    109: {'name':'Chase Field',              'hr_L':1.00,'hr_R':1.03,'lat':33.445,'lon':-112.067,'roof':True, 'orientation':0},
    110: {'name':'Camden Yards',             'hr_L':1.07,'hr_R':1.03,'lat':39.284,'lon':-76.622, 'roof':False,'orientation':83},
    111: {'name':'Fenway Park',              'hr_L':0.93,'hr_R':1.04,'lat':42.347,'lon':-71.097, 'roof':False,'orientation':93},
    112: {'name':'Wrigley Field',            'hr_L':1.05,'hr_R':1.01,'lat':41.948,'lon':-87.656, 'roof':False,'orientation':93},
    113: {'name':'Great American Ball Park', 'hr_L':1.16,'hr_R':1.14,'lat':39.098,'lon':-84.507, 'roof':False,'orientation':352},
    114: {'name':'Progressive Field',        'hr_L':0.96,'hr_R':0.98,'lat':41.496,'lon':-81.685, 'roof':False,'orientation':340},
    115: {'name':'Coors Field',              'hr_L':1.34,'hr_R':1.38,'lat':39.756,'lon':-104.994,'roof':False,'orientation':335},
    116: {'name':'Comerica Park',            'hr_L':0.91,'hr_R':0.94,'lat':42.339,'lon':-83.048, 'roof':False,'orientation':27},
    117: {'name':'Minute Maid Park',         'hr_L':0.97,'hr_R':1.00,'lat':29.757,'lon':-95.356, 'roof':True, 'orientation':0},
    118: {'name':'Kauffman Stadium',         'hr_L':0.93,'hr_R':0.91,'lat':39.051,'lon':-94.480, 'roof':False,'orientation':5},
    119: {'name':'Dodger Stadium',           'hr_L':0.99,'hr_R':1.03,'lat':34.074,'lon':-118.240,'roof':False,'orientation':25},
    120: {'name':'Nationals Park',           'hr_L':1.01,'hr_R':0.97,'lat':38.873,'lon':-77.007, 'roof':False,'orientation':355},
    121: {'name':'Citi Field',               'hr_L':0.96,'hr_R':0.92,'lat':40.757,'lon':-73.846, 'roof':False,'orientation':358},
}

BASE = 'https://statsapi.mlb.com/api/v1'

def mlb(path, params=None):
    r = requests.get(BASE + path, params=params, timeout=15)
    r.raise_for_status()
    return r.json()

def fetch_weather(lat, lon, game_dt):
    if not lat: return {}
    try:
        dt = datetime.fromisoformat(game_dt.replace('Z','+00:00'))
        r = requests.get('https://api.open-meteo.com/v1/forecast', params={
            'latitude': lat, 'longitude': lon,
            'hourly': 'temperature_2m,windspeed_10m,winddirection_10m',
            'windspeed_unit': 'mph', 'temperature_unit': 'fahrenheit',
            'timezone': 'auto',
            'start_date': dt.strftime('%Y-%m-%d'),
            'end_date':   dt.strftime('%Y-%m-%d'),
        }, timeout=10)
        h = r.json().get('hourly', {})
        times = h.get('time', [])
        target = dt.strftime('%Y-%m-%dT%H:00')
        idx = times.index(target) if target in times else 0
        return {
            'temp_f':       round(h['temperature_2m'][idx], 1),
            'wind_mph':     round(h['windspeed_10m'][idx], 1),
            'wind_dir_deg': h['winddirection_10m'][idx],
        }
    except: return {}

def wind_adj(wind_mph, wind_dir, park_orient, roof):
    if roof or wind_mph < 3: return 1.0
    cf = (park_orient + 180) % 360
    delta = abs(wind_dir - cf)
    if delta > 180: delta = 360 - delta
    return round(max(0.85, min(1.20, 1.0 + math.cos(math.radians(delta)) * wind_mph * 0.007)), 3)

def mlb_batter_stats(pid):
    if not pid: return {}
    def _fetch(season):
        try:
            splits = mlb(f'/people/{pid}/stats', {'stats':'season','group':'hitting','season':season})
            s = (splits.get('stats',[{}])[0].get('splits',[{}]) or [{}])[0].get('stat',{})
            ab = int(s.get('atBats') or 0)
            if ab == 0: return None
            slg = float(s.get('slg') or 0)
            avg = float(s.get('avg') or 0)
            hr  = int(s.get('homeRuns') or 0)
            return {'avg':s.get('avg'),'slg':s.get('slg'),'ops':s.get('ops'),
                    'iso':round(slg-avg,3),'hr':hr,'hr_per_ab':round(hr/ab,4)}
        except: return None
    return _fetch(2026) or _fetch(2025) or {}

def mlb_pitcher_stats(pid):
    if not pid: return {}
    def _fetch(season):
        try:
            splits = mlb(f'/people/{pid}/stats', {'stats':'season','group':'pitching','season':season})
            s = (splits.get('stats',[{}])[0].get('splits',[{}]) or [{}])[0].get('stat',{})
            bf = int(s.get('battersFaced') or 0)
            if bf == 0: return None
            return {'era':s.get('era'),'whip':s.get('whip'),'hr9':s.get('homeRunsPer9'),
                    'k_pct':round(int(s.get('strikeOuts') or 0)/max(bf,1),3)}
        except: return None
    return _fetch(2026) or _fetch(2025) or {}

def _sc_batter_df(name):
    if not PYBASEBALL: return None, None
    parts = name.split()
    ids = playerid_lookup(parts[-1], parts[0])
    if ids.empty: return None, None
    pid = int(ids.iloc[0]['key_mlbam'])
    df26 = statcast_batter(SC_START_2026, SC_END_2026, player_id=pid)
    df25 = statcast_batter(SC_START_2025, SC_END_2025, player_id=pid)
    n26 = len(df26) if df26 is not None and not df26.empty else 0
    n25 = len(df25) if df25 is not None and not df25.empty else 0
    if n26 >= SC_MIN_PA_2026: return df26, '2026'
    elif n26 > 0 and n25 > 0: return pd.concat([df25, df26], ignore_index=True), '2025+2026'
    elif n25 > 0: return df25, '2025'
    return None, None

def _sc_pitcher_df(name):
    if not PYBASEBALL: return None, None
    parts = name.split()
    ids = playerid_lookup(parts[-1], parts[0])
    if ids.empty: return None, None
    pid = int(ids.iloc[0]['key_mlbam'])
    df26 = statcast_pitcher(SC_START_2026, SC_END_2026, player_id=pid)
    df25 = statcast_pitcher(SC_START_2025, SC_END_2025, player_id=pid)
    n26 = len(df26) if df26 is not None and not df26.empty else 0
    n25 = len(df25) if df25 is not None and not df25.empty else 0
    if n26 >= SC_MIN_PA_2026: return df26, '2026'
    elif n26 > 0 and n25 > 0: return pd.concat([df25, df26], ignore_index=True), '2025+2026'
    elif n25 > 0: return df25, '2025'
    return None, None

def sc_batter(name):
    if not (PYBASEBALL and USE_STATCAST): return {}
    try:
        df, _ = _sc_batter_df(name)
        if df is None or df.empty: return {}
        pa = len(df)
        result = {
            'barrel_pct':    round(df[df['launch_speed_angle']==6].shape[0]/max(pa,1)*100,1) if 'launch_speed_angle' in df.columns else None,
            'hard_hit_pct':  round(df[df['launch_speed']>=95].shape[0]/max(pa,1)*100,1) if 'launch_speed' in df.columns else None,
            'fb_pct':        round(df[df['bb_type']=='fly_ball'].shape[0]/max(pa,1)*100,1) if 'bb_type' in df.columns else None,
            'avg_exit_velo': round(df['launch_speed'].dropna().mean(),1) if 'launch_speed' in df.columns else None,
            'hr_per_pa':     round(df[df['events']=='home_run'].shape[0]/max(pa,1),4) if 'events' in df.columns else None,
        }
        return {k:v for k,v in result.items() if v is not None}
    except: return {}

def sc_pitcher(name):
    if not (PYBASEBALL and USE_STATCAST): return {}
    try:
        df, _ = _sc_pitcher_df(name)
        if df is None or df.empty: return {}
        bf = len(df)
        hrs = df[df['events']=='home_run'].shape[0] if 'events' in df.columns else 0
        result = {
            'hr_per_bf':            round(hrs/max(bf,1),4),
            'hard_hit_pct_allowed': round(df[df['launch_speed']>=95].shape[0]/max(bf,1)*100,1) if 'launch_speed' in df.columns else None,
        }
        return {k:v for k,v in result.items() if v is not None}
    except: return {}

def build_sp(sp_dict, label):
    sp_id = sp_dict.get('id')
    sp_name = sp_dict.get('fullName', 'TBD')
    print(f'  {label}: {sp_name}')
    stats = {**mlb_pitcher_stats(sp_id), **sc_pitcher(sp_name)}
    time.sleep(0.3)
    return {'name': sp_name, 'id': sp_id, **stats}

def build_lineup(raw, team_id):
    if not raw:
        print('  Lineup not yet posted — using active position player roster')
        roster = mlb(f'/teams/{team_id}/roster', {'rosterType':'active'}).get('roster',[])
        raw = [
            {'id': p['person']['id'], 'fullName': p['person']['fullName']}
            for p in roster
            if p.get('position',{}).get('code') not in ('1','P')
        ][:13]
    batters = []
    for p in raw:
        pid   = p.get('id')
        pname = p.get('fullName', p.get('name',''))
        # Fetch batting hand from player profile
        hand = 'R'
        try:
            info = mlb(f'/people/{pid}')
            hand = info.get('people',[{}])[0].get('batSide',{}).get('code','R')
        except: pass
        stats = {**mlb_batter_stats(pid), **sc_batter(pname)}
        batters.append({'id':pid, 'name':pname, 'hand':hand, 'stats':stats})
        time.sleep(0.2)
    return batters

# Fetch schedule
print(f'Fetching schedule for {GAME_DATE}...')
sched = mlb('/schedule', {'sportId':1,'date':GAME_DATE,'hydrate':'probablePitcher,lineups,team,venue'})
raw_games = [g for d in sched.get('dates',[]) for g in d.get('games',[])]
print(f'✓ {len(raw_games)} games found')

payload_games = []
for g in raw_games:
    home      = g['teams']['home']
    away      = g['teams']['away']
    home_id   = home['team']['id']
    away_id   = away['team']['id']
    home_abbr = home['team'].get('abbreviation','?')
    away_abbr = away['team'].get('abbreviation','?')
    park      = PARKS.get(home_id, {})
    matchup   = f'{away_abbr} @ {home_abbr}'
    print(f'\n── {matchup} ──')

    home_sp_raw = home.get('probablePitcher', {})
    away_sp_raw = away.get('probablePitcher', {})
    home_lineup_raw = g.get('lineups', {}).get('homePlayers', [])
    away_lineup_raw = g.get('lineups', {}).get('awayPlayers', [])

    weather = {}
    if not park.get('roof'):
        weather = fetch_weather(park.get('lat'), park.get('lon'), g.get('gameDate',''))
        if weather:
            weather['wind_hr_adj'] = wind_adj(
                weather['wind_mph'], weather['wind_dir_deg'],
                park.get('orientation', 0), False
            )
            print(f'  Weather: {weather["temp_f"]}°F, wind {weather["wind_mph"]} mph → HR adj {weather["wind_hr_adj"]}')
        time.sleep(0.4)

    # pitcher_facing_home_batters = away SP (faces HOME batters)
    # pitcher_facing_away_batters = home SP (faces AWAY batters)
    sp_facing_home = build_sp(away_sp_raw, f'SP facing {home_abbr} batters (away pitcher)')
    sp_facing_away = build_sp(home_sp_raw, f'SP facing {away_abbr} batters (home pitcher)')

    print(f'  Building {home_abbr} lineup...')
    home_batters = build_lineup(home_lineup_raw, home_id)
    print(f'  Building {away_abbr} lineup...')
    away_batters = build_lineup(away_lineup_raw, away_id)

    payload_games.append({
        'matchup':  matchup,
        'game_date': g.get('gameDate','')[:16],
        'venue': {
            'name':        park.get('name', g.get('venue',{}).get('name','')),
            'hr_factor_L': park.get('hr_L', 1.0),
            'hr_factor_R': park.get('hr_R', 1.0),
        },
        'weather': weather,
        'pitcher_facing_home_batters': sp_facing_home,
        'pitcher_facing_away_batters': sp_facing_away,
        'home_batters': home_batters,
        'away_batters': away_batters,
    })

print(f'\n✓ Data collection complete — {len(payload_games)} games')

In [ ]:
# ── CELL 4: Score all batters in pure Python ──────────────────────────────────
#
# Composite HR score = weighted blend of four sub-scores:
#   batter_score   (40%) — power metrics: barrel%, hard-hit%, ISO, HR rate, exit velo
#   pitcher_score  (35%) — opposing SP vulnerability: HR/9, HR/BF, hard-hit allowed
#   park_score     (15%) — park HR factor for the batter's handedness
#   weather_score  (10%) — wind direction/speed adjustment for outdoor parks
#
# All sub-scores are 0–100. Final score is clamped to 0–100.

def norm(v, lo, hi):
    """Linearly map v from [lo, hi] → [0, 100], clamped."""
    if hi == lo: return 50
    return max(0, min(100, (v - lo) / (hi - lo) * 100))

def batter_score(stats):
    """0–100: how likely is this batter to hit a HR based on power metrics."""
    pts, wts = [], []

    if 'barrel_pct' in stats:      # best single predictor
        pts.append(norm(stats['barrel_pct'], 2, 18));    wts.append(2.5)
    if 'hard_hit_pct' in stats:
        pts.append(norm(stats['hard_hit_pct'], 20, 55)); wts.append(2.0)
    if 'iso' in stats:
        pts.append(norm(stats['iso'], 0.05, 0.30));      wts.append(2.0)
    if 'hr_per_pa' in stats:
        pts.append(norm(stats['hr_per_pa'], 0.01, 0.08)); wts.append(1.5)
    elif 'hr_per_ab' in stats:
        pts.append(norm(stats['hr_per_ab'], 0.01, 0.10)); wts.append(1.5)
    if 'avg_exit_velo' in stats:
        pts.append(norm(stats['avg_exit_velo'], 82, 95)); wts.append(1.5)
    if 'fb_pct' in stats:
        pts.append(norm(stats['fb_pct'], 20, 50));        wts.append(1.0)
    if 'slg' in stats and stats['slg']:
        try: pts.append(norm(float(stats['slg']), 0.30, 0.65)); wts.append(0.5)
        except: pass

    return round(sum(p*w for p,w in zip(pts,wts)) / sum(wts)) if pts else 35

def pitcher_vuln_score(stats):
    """0–100: how vulnerable is this pitcher to giving up HRs."""
    pts, wts = [], []

    if 'hr_per_bf' in stats:
        pts.append(norm(stats['hr_per_bf'], 0.01, 0.06)); wts.append(3.0)
    if 'hr9' in stats and stats['hr9']:
        try: pts.append(norm(float(stats['hr9']), 0.5, 2.5)); wts.append(2.5)
        except: pass
    if 'hard_hit_pct_allowed' in stats:
        pts.append(norm(stats['hard_hit_pct_allowed'], 25, 55)); wts.append(2.0)
    if 'era' in stats and stats['era']:
        try: pts.append(norm(float(stats['era']), 2.5, 6.0)); wts.append(1.0)
        except: pass

    return round(sum(p*w for p,w in zip(pts,wts)) / sum(wts)) if pts else 50

def park_score(venue, hand):
    """0–100: how HR-friendly is this park for this batter's hand."""
    key = 'hr_factor_L' if hand == 'L' else 'hr_factor_R'
    return round(norm(venue.get(key, 1.0), 0.78, 1.40))

def weather_score(weather):
    """0–100: wind boost converted to a score (1.0 adj = 50, 1.20 = 100, 0.85 = 0)."""
    adj = weather.get('wind_hr_adj', 1.0)
    return round(norm(adj, 0.85, 1.20))

def composite_score(bs, ps, pk, ws):
    return round(max(0, min(100, bs*0.40 + ps*0.35 + pk*0.15 + ws*0.10)))

def confidence(bs, n_stats):
    if n_stats >= 5 and bs >= 65: return 'high'
    if n_stats >= 3 and bs >= 45: return 'medium'
    return 'low'

def form_note(stats):
    notes = []
    if stats.get('barrel_pct', 0) >= 12: notes.append(f"barrel% {stats['barrel_pct']}")
    if stats.get('avg_exit_velo', 0) >= 92: notes.append(f"EV {stats['avg_exit_velo']} mph")
    if stats.get('iso', 0) >= 0.220: notes.append(f"ISO {stats['iso']}")
    if not notes and stats.get('slg'): notes.append(f"SLG {stats['slg']}")
    return ', '.join(notes) if notes else '—'

def reasoning(name, stats, sp_name, sp_stats, bs, ps, pk):
    parts = []
    if stats.get('barrel_pct'): parts.append(f"barrel% {stats['barrel_pct']}")
    if stats.get('hard_hit_pct'): parts.append(f"hard-hit% {stats['hard_hit_pct']}")
    if stats.get('iso'): parts.append(f"ISO {stats['iso']}")
    batter_str = ', '.join(parts[:3]) or 'limited data'

    sp_parts = []
    if sp_stats.get('hr9'): sp_parts.append(f"HR/9 {sp_stats['hr9']}")
    if sp_stats.get('era'): sp_parts.append(f"ERA {sp_stats['era']}")
    sp_str = ', '.join(sp_parts) or 'limited data'

    return (f"{name} posts {batter_str} (batter score {bs}). "
            f"{sp_name} shows {sp_str} (vulnerability score {ps}); "
            f"park factor contributes {pk}/100.")

# ── Score every batter in every game ─────────────────────────────────────────
candidates = []

for game in payload_games:
    venue   = game['venue']
    weather = game['weather']
    matchup = game['matchup']
    park_name = venue['name']

    # home SP = pitcher_facing_away_batters; away SP = pitcher_facing_home_batters
    away_abbr = matchup.split(' @ ')[0]
    home_abbr = matchup.split(' @ ')[1]

    for team_label, batters, sp in [
        (home_abbr, game['home_batters'], game['pitcher_facing_home_batters']),
        (away_abbr, game['away_batters'], game['pitcher_facing_away_batters']),
    ]:
        ps = pitcher_vuln_score(sp)
        ws = weather_score(weather)

        for b in batters:
            stats = b.get('stats', {})
            hand  = b.get('hand', 'R')
            bs    = batter_score(stats)
            pk    = park_score(venue, hand)
            score = composite_score(bs, ps, pk, ws)
            n_stats = sum(1 for k in ['barrel_pct','hard_hit_pct','iso','hr_per_pa',
                                       'avg_exit_velo','fb_pct','slg'] if k in stats)
            candidates.append({
                'player':       b['name'],
                'team':         team_label,
                'hand':         hand,
                'game':         matchup,
                'park':         park_name,
                'score':        score,
                'confidence':   confidence(bs, n_stats),
                'batter_score': bs,
                'pitcher_score': ps,
                'bullpen_score': 50,   # placeholder — no bullpen data in schedule API
                'park_score':   pk,
                'sp_name':      sp['name'],
                'sp_era':       str(sp.get('era', '—')),
                'recent_form':  form_note(stats),
                'reasoning':    reasoning(b['name'], stats, sp['name'], sp, bs, ps, pk),
            })

predictions = sorted(candidates, key=lambda x: -x['score'])[:TOP_N_PLAYERS]

print(f'\n✓ Scored {len(candidates)} batters across {len(payload_games)} games')
print(f'  Top {len(predictions)} predictions:')
for i, p in enumerate(predictions[:10], 1):
    print(f'  {i:2d}. {p["player"]:28s} {p["team"]:4s}  score={p["score"]:3d}  conf={p["confidence"]}')

In [ ]:
# ── CELL 5: Save predictions.json and push to GitHub Pages ────────────────────
import json, subprocess
from pathlib import Path
from datetime import datetime

output = {
    'game_date':    GAME_DATE,
    'generated_at': datetime.utcnow().isoformat() + 'Z',
    'games_count':  len(payload_games),
    'predictions':  predictions,
}

repo = Path(REPO_PATH).resolve()

# Write predictions.json
(repo / 'predictions.json').write_text(json.dumps(output, indent=2))
print(f'✓ predictions.json saved ({len(predictions)} players)')

# Archive to history/
history_dir = repo / 'history'
history_dir.mkdir(exist_ok=True)
(history_dir / f'{GAME_DATE}.json').write_text(json.dumps(output, indent=2))
print(f'✓ history/{GAME_DATE}.json saved')

if AUTO_PUSH:
    cmds = [
        ['git', 'add', 'predictions.json', 'history/'],
        ['git', 'commit', '-m', f'predictions {GAME_DATE}'],
        ['git', 'push'],
    ]
    for cmd in cmds:
        result = subprocess.run(cmd, cwd=repo, capture_output=True, text=True)
        if result.returncode != 0 and 'nothing to commit' not in result.stdout:
            print(f'  ⚠ {" ".join(cmd)}: {result.stderr.strip()}')
        else:
            print(f'  ✓ {" ".join(cmd)}')
    print('\n🎯 Site updated — predictions are live on GitHub Pages')
else:
    print('\nAUTO_PUSH is False — run git push manually to update the site')

In [ ]:
# ── CELL 6: Log results after games complete ──────────────────────────────────
# Run this the evening after games are played.
# The predictions list is already in memory from Cells 3–4.
# Just run this cell and enter y/n/d/s for each player.

import json
from pathlib import Path

RESULTS_FILE = Path(REPO_PATH) / 'hr_scout_results.json'
log = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.exists() else {}

print(f'Logging results for {GAME_DATE}')
print('Enter: y = hit HR   n = no HR   d = did not play   s = skip\n')

entries = []
for p in sorted(predictions, key=lambda x: -x['score']):
    ans = input(f"  {p['player']:28s} {p['team']:4s} score={p['score']:3d} → ").strip().lower()
    entries.append({
        'player':  p['player'],
        'team':    p['team'],
        'score':   p['score'],
        'hr_hit':  ans == 'y',
        'played':  ans != 'd',
        'skipped': ans == 's',
    })

played  = [e for e in entries if e['played'] and not e['skipped']]
hr_hits = [e for e in played  if e['hr_hit']]

buckets = {'70+': [], '60-69': [], '50-59': [], '<50': []}
for e in played:
    b = '70+' if e['score']>=70 else '60-69' if e['score']>=60 else '50-59' if e['score']>=50 else '<50'
    buckets[b].append(e['hr_hit'])

result = {
    'date':         GAME_DATE,
    'total':        len(entries),
    'played':       len(played),
    'hr_count':     len(hr_hits),
    'hit_rate_pct': round(len(hr_hits)/max(len(played),1)*100, 1),
    'hr_players':   [e['player'] for e in hr_hits],
    'by_bucket':    {k: {'predicted':len(v),'hr':sum(v),
                         'rate_pct':round(sum(v)/max(len(v),1)*100,1)}
                     for k,v in buckets.items() if v},
    'entries':      entries,
}
log[GAME_DATE] = result
RESULTS_FILE.write_text(json.dumps(log, indent=2))

# Push updated results file
if AUTO_PUSH:
    subprocess.run(['git','add','hr_scout_results.json'], cwd=Path(REPO_PATH).resolve())
    subprocess.run(['git','commit','-m',f'results {GAME_DATE}'], cwd=Path(REPO_PATH).resolve())
    subprocess.run(['git','push'], cwd=Path(REPO_PATH).resolve())

print(f'\n✓ {result["hr_count"]}/{result["played"]} hit HRs ({result["hit_rate_pct"]}%)')
for bucket, stats in result['by_bucket'].items():
    bar = '█'*stats['hr'] + '░'*(stats['predicted']-stats['hr'])
    print(f'  {bucket:6s} {bar:30s} {stats["hr"]}/{stats["predicted"]} ({stats["rate_pct"]}%)')

In [ ]:
# ── CELL 7: Calibration report ────────────────────────────────────────────────
import json
from pathlib import Path

RESULTS_FILE = Path(REPO_PATH) / 'hr_scout_results.json'
if not RESULTS_FILE.exists():
    print('No results logged yet. Run Cell 6 after games complete.')
else:
    log = json.loads(RESULTS_FILE.read_text())
    print('═══ HR Scout Calibration Report ═══\n')
    totals = {'70+':{'p':0,'h':0},'60-69':{'p':0,'h':0},'50-59':{'p':0,'h':0},'<50':{'p':0,'h':0}}
    for d, entry in sorted(log.items()):
        print(f"{d}:  {entry['hr_count']}/{entry['played']} HR ({entry['hit_rate_pct']}%)  —  {', '.join(entry['hr_players']) or 'none'}")
        for bucket, stats in entry.get('by_bucket', {}).items():
            if bucket in totals:
                totals[bucket]['p'] += stats['predicted']
                totals[bucket]['h'] += stats['hr']
    print('\n── Cumulative hit rate by score bucket ──')
    for bucket, t in totals.items():
        if t['p'] > 0:
            rate = round(t['h']/t['p']*100, 1)
            bar  = '█'*int(rate/2) + '░'*(50-int(rate/2))
            print(f'  {bucket:6s}  {bar}  {rate:5.1f}%  ({t["h"]}/{t["p"]})')
    print('\nMLB baseline HR rate: ~3% per PA')